# tanque-cheio · 07 — modelo final em GGUF para a demo (M19)

Funde o adapter v2 (`ft2`, M15/M17: EX de 80,7% no teste) no
Qwen2.5-Coder-1.5B-Instruct, converte para **GGUF Q4_K_M** com o llama.cpp e confere o
resultado antes de publicar (D36):

1. `merge_and_unload()` do adapter, pesos salvos em **fp16** (safetensors);
2. `convert_hf_to_gguf.py` do llama.cpp (tag fixada) → GGUF f16;
3. quantização Q4_K_M com a função oficial `llama_model_quantize` da API C do
   llama.cpp, chamada pelo `llama-cpp-python` (wheel CPU pré-compilado, a **mesma
   versão do app** e o mesmo llama.cpp da tag acima): nada é compilado;
4. tamanho dos arquivos e SHA-256;
5. conferência rápida em **3 perguntas do dev2** (uma por nível): a SQL do GGUF
   (llama-cpp-python, `create_chat_completion`, gulosa, contexto 2048 — o mesmo
   gerador do app) contra a do modelo fundido em fp16 (transformers), com o número
   de SQLs idênticas e de resultados que **batem na execução**;
6. model card e, só com `PUSH_TO_HUB = True` e o secret `HF_TOKEN`, publicação em
   `belasoares/tanque-cheio-qwen2.5-coder-1.5b-gguf`.

**Hardware:** CPU basta (sessão CPU do Kaggle: 4 vCPU, ~30 GB de RAM). Fundir em fp32
ocupa ~7 GB de RAM; o GGUF f16 tem ~3,1 GB e o Q4_K_M ~1,0 GB. Com GPU, o modelo de
referência roda em fp16 na GPU; sem GPU, roda em fp32 na CPU com os pesos já
arredondados para fp16 (os mesmos que foram salvos). Tempo estimado em CPU: ~20 min.

**No Kaggle:** Internet on, sem acelerador. Para publicar: Add-ons → Secrets → `HF_TOKEN`
(token de escrita do HF) e `PUSH_TO_HUB = True`. Passos em `docs/demo.md`.

As saídas ficam em `/kaggle/working`: o `.gguf` Q4_K_M, `README.md` (model card) e
`export-info.json`. O modelo fundido e o GGUF f16 ficam em `/tmp`, fora da saída.

In [ ]:
# Parâmetros: edite só esta célula.
REPO_REF = "main"  # branch, tag ou commit do tanque-cheio (o app/ precisa existir nele)
BASE_MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
BASE_REVISION = "2e1fd397ee46e1388853d2af2c993145b0f1098a"  # o mesmo do M15/M17
ADAPTER_ID = "belasoares/tanque-cheio-qwen2.5-coder-1.5b-lora-v2"  # ft2 (M15)
ADAPTER_REVISION = "beeb19b5b4331ff048c6be0f03b9d53476d509ab"  # o mesmo do M17
# llama.cpp: a tag que o llama-cpp-python 0.3.36 embute (vendor/llama.cpp), para o
# conversor e o quantizador serem do mesmo llama.cpp que roda no app.
LLAMA_CPP_REF = "b11312"
LLAMA_CPP_COMMIT_EXPECTED = "0c1e57098bba43ac29e6e3b677cdceebdd22334f"
LLAMA_CPP_PYTHON = "0.3.36"  # a mesma versão de app/requirements.txt
QUANT = "Q4_K_M"  # D36
GGUF_NAME = "tanque-cheio-qwen2.5-coder-1.5b-q4_k_m.gguf"  # o nome que o app baixa
CHECK_DATASET = "dev2"  # conferência só no dev (regra 0.6); o teste não é usado aqui
MAX_NEW_TOKENS = 384  # D31
N_CTX = 2048  # o mesmo contexto do app
PUSH_TO_HUB = False  # publicar exige aprovação (regra 0.5): a Isabela liga
HF_REPO = "belasoares/tanque-cheio-qwen2.5-coder-1.5b-gguf"
OUT_DIR = "/kaggle/working"  # vira a saída do notebook no Kaggle
WORK_DIR = "/tmp/export"  # modelo fundido e GGUF f16 (~3 GB cada), fora da saída
REPO_DIR = "/tmp/tanque-cheio"
LLAMA_CPP_DIR = "/tmp/llama.cpp"

In [ ]:
# Instala o pacote `tanque` no REPO_REF (versões do M6/M17), o llama-cpp-python (wheel CPU
# oficial, sem compilar) e clona o tanque-cheio e o llama.cpp nos refs fixados.
import os
import subprocess
import sys

# Regra 0.8: uma GPU só, antes de importar o torch (sem GPU, não muda nada).
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

REPO_URL = "https://github.com/belasoaress/tanque-cheio"
LLAMA_CPP_URL = "https://github.com/ggml-org/llama.cpp"
LLAMA_CPP_WHEELS = "https://abetlen.github.io/llama-cpp-python/whl/cpu"
# As mesmas versões do fine-tuning e do teste (uv.lock); torch vem da imagem.
PINS = [
    "transformers==5.21.0",
    "peft==0.21.2",
    "accelerate==1.15.0",
    # O conversor do llama.cpp importa o sentencepiece antes de cair no tokenizer BPE do
    # Qwen; a faixa é a do requirements do llama.cpp (>=0.1.98,<0.3.0).
    "sentencepiece==0.2.1",
]


def sh(*cmd: str) -> None:
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)


sh(sys.executable, "-m", "pip", "install", "-q", f"tanque @ git+{REPO_URL}@{REPO_REF}", *PINS)
sh(
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--only-binary=llama-cpp-python",
    "--extra-index-url",
    LLAMA_CPP_WHEELS,
    f"llama-cpp-python=={LLAMA_CPP_PYTHON}",
)
# A imagem do Kaggle traz um torchao antigo (0.10) e o peft se recusa a carregar o
# adapter quando ele está instalado. O projeto não usa torchao: remove.
sh(sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao")
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    sh("git", "clone", "--quiet", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "--quiet", "origin", REPO_REF)
sh("git", "-C", REPO_DIR, "checkout", "--quiet", "FETCH_HEAD")
REPO_COMMIT = subprocess.check_output(
    ["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True
).strip()
os.environ["TANQUE_ROOT"] = REPO_DIR  # antes de importar `tanque`
print("tanque-cheio em", REPO_COMMIT)

# Só o conversor do llama.cpp (Python) e o gguf-py ao lado dele; nada é compilado. O
# requirements.txt do llama.cpp NÃO é instalado: ele fixa torch e transformers 4 e
# quebraria o resto. O conversor usa o gguf-py da própria tag (numpy, pyyaml, tqdm e
# requests já vêm na imagem).
if not os.path.isdir(os.path.join(LLAMA_CPP_DIR, ".git")):
    sh(
        "git",
        "clone",
        "--quiet",
        "--depth",
        "1",
        "--branch",
        LLAMA_CPP_REF,
        LLAMA_CPP_URL,
        LLAMA_CPP_DIR,
    )
LLAMA_CPP_COMMIT = subprocess.check_output(
    ["git", "-C", LLAMA_CPP_DIR, "rev-parse", "HEAD"], text=True
).strip()
assert LLAMA_CPP_COMMIT == LLAMA_CPP_COMMIT_EXPECTED, (LLAMA_CPP_REF, LLAMA_CPP_COMMIT)
print("llama.cpp em", LLAMA_CPP_REF, LLAMA_CPP_COMMIT)

In [ ]:
import llama_cpp
import torch
import transformers
from peft.import_utils import is_torchao_available

# Falha cedo se o peft não puder carregar adapters (torchao incompatível).
is_torchao_available()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(
    "torch",
    torch.__version__,
    "| transformers",
    transformers.__version__,
    "| llama-cpp-python",
    llama_cpp.__version__,
    "| dispositivo",
    DEVICE,
    "| CPUs",
    os.cpu_count(),
)

In [ ]:
# Banco DuckDB do snapshot versionado (D25), para executar as SQLs da conferência
# (~6 s, sem download). Equivale a `tanque data build --from-parquet ...`.
sh(
    sys.executable,
    "-c",
    "from tanque.cli import app; app()",
    "data",
    "build",
    "--from-parquet",
    os.path.join(REPO_DIR, "snapshot", "coletas-2026-09.parquet"),
)

In [ ]:
# 1. Funde o adapter v2 no modelo base e salva em fp16.
import gc
import json
import time
from pathlib import Path

from tanque.infer import local

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
MERGED_DIR = Path(WORK_DIR) / "merged-fp16"
timings = {}

t0 = time.perf_counter()
# Na CPU, funde em fp32 (fp16 na CPU é lento e menos preciso); na GPU, em fp16 como no M17.
model, tokenizer = local.load_model(
    BASE_MODEL_ID,
    ADAPTER_ID,
    dtype="float16" if DEVICE == "cuda" else "float32",
    device=DEVICE,
    revision=BASE_REVISION,
    adapter_revision=ADAPTER_REVISION,
    merge_adapter=True,  # merge_and_unload(): um modelo comum, sem PeftModel
)
model = model.to(torch.float16)
model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
if DEVICE == "cpu":
    # Referência da conferência: os mesmos pesos fp16 salvos, calculados em fp32 na CPU.
    model = model.to(torch.float32)
timings["merge_s"] = round(time.perf_counter() - t0, 1)
print(f"fundido e salvo em {MERGED_DIR} ({timings['merge_s']} s)")
print(sorted(p.name for p in MERGED_DIR.iterdir()))

In [ ]:
# 2. Converte para GGUF f16 com o conversor do llama.cpp (copia o chat template do
# tokenizer para os metadados do GGUF, que o llama-cpp-python usa no app).
F16_GGUF = Path(WORK_DIR) / "tanque-cheio-qwen2.5-coder-1.5b-f16.gguf"
t0 = time.perf_counter()
sh(
    sys.executable,
    os.path.join(LLAMA_CPP_DIR, "convert_hf_to_gguf.py"),
    str(MERGED_DIR),
    "--outtype",
    "f16",
    "--outfile",
    str(F16_GGUF),
    "--model-name",
    "tanque-cheio-qwen2.5-coder-1.5b",  # general.name do GGUF (senão, o nome da pasta)
)
timings["convert_s"] = round(time.perf_counter() - t0, 1)
print(f"GGUF f16 em {timings['convert_s']} s")

In [ ]:
# 3. Quantiza para Q4_K_M com `llama_model_quantize` (API C do llama.cpp, a mesma do
# binário llama-quantize), pelo llama-cpp-python: sem compilar nada.
import ctypes

Q_GGUF = Path(OUT_DIR) / GGUF_NAME
params = llama_cpp.llama_model_quantize_default_params()
params.ftype = getattr(llama_cpp, f"LLAMA_FTYPE_MOSTLY_{QUANT}")
params.nthread = os.cpu_count() or 4
t0 = time.perf_counter()
rc = llama_cpp.llama_model_quantize(
    str(F16_GGUF).encode(), str(Q_GGUF).encode(), ctypes.byref(params)
)
assert rc == 0, f"llama_model_quantize devolveu {rc}"
timings["quantize_s"] = round(time.perf_counter() - t0, 1)
print(f"{QUANT} em {timings['quantize_s']} s")

In [ ]:
# 4. Tamanhos e SHA-256.
import hashlib


def sha256_of(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while chunk := f.read(1 << 20):
            h.update(chunk)
    return h.hexdigest()


def gib(n: int) -> float:
    return round(n / 2**30, 3)


merged_bytes = sum(p.stat().st_size for p in MERGED_DIR.glob("*.safetensors"))
sizes = {
    "merged_fp16_safetensors_gib": gib(merged_bytes),
    "gguf_f16_gib": gib(F16_GGUF.stat().st_size),
    "gguf_q_gib": gib(Q_GGUF.stat().st_size),
    "gguf_q_bytes": Q_GGUF.stat().st_size,
}
GGUF_SHA256 = sha256_of(Q_GGUF)
print(json.dumps(sizes, indent=2), "\nsha256", GGUF_SHA256)

In [ ]:
# 5. Conferência rápida: 3 perguntas do dev2 (a 1ª de cada nível), GGUF × fp16.
from llama_cpp import Llama
from llama_cpp.llama_chat_format import Jinja2ChatFormatter

from tanque import config, prompting
from tanque.eval.compare import results_match
from tanque.sql import execute

# O gerador do app (app/tanque_demo.py, no mesmo REPO_REF): mesma chamada do Space.
sys.path.insert(0, os.path.join(REPO_DIR, "app"))
import tanque_demo  # noqa: E402

lines = config.resolve_path(f"datasets/{CHECK_DATASET}.jsonl").read_text(encoding="utf-8")
items = [json.loads(line) for line in lines.splitlines() if line.strip()]
picked = [
    next(it for it in items if it["difficulty"] == lvl) for lvl in ("facil", "medio", "dificil")
]

hf_generate = local.chat_generate(local.HFGenerator(model, tokenizer, MAX_NEW_TOKENS), tokenizer)
llm = Llama(model_path=str(Q_GGUF), n_ctx=N_CTX, n_threads=os.cpu_count(), seed=0, verbose=False)
gguf_generate = tanque_demo.llama_generator(llm, max_tokens=MAX_NEW_TOKENS)

# O texto do prompt do GGUF (chat template dos metadados) é o mesmo do transformers?
formatter = Jinja2ChatFormatter(
    template=llm.metadata["tokenizer.chat_template"],
    eos_token=llm._model.token_get_text(llm.token_eos()),
    bos_token="",
)
msgs = prompting.build_prompt(picked[0]["question"])
same_prompt = formatter(messages=msgs).prompt == local.render_prompt(tokenizer, msgs).text
print(
    "chat_format do llama-cpp-python:", llm.chat_format, "| prompt idêntico ao do HF:", same_prompt
)


def norm(sql: str) -> str:
    return " ".join(sql.split()).rstrip(";").lower()


check = []
for it in picked:
    msgs = prompting.build_prompt(it["question"])
    t0 = time.perf_counter()
    hf_sql = prompting.extract_sql(hf_generate(msgs).text)
    hf_s = time.perf_counter() - t0
    t0 = time.perf_counter()
    q_sql = prompting.extract_sql(gguf_generate(msgs).text)
    q_s = time.perf_counter() - t0
    gold, hf_res, q_res = (execute.run(s) for s in (it["sql"], hf_sql, q_sql))
    om = bool(it.get("order_matters"))
    row = {
        "id": it["id"],
        "difficulty": it["difficulty"],
        "same_sql": norm(hf_sql) == norm(q_sql),
        "same_result": (not hf_res.error and not q_res.error) and results_match(hf_res, q_res, om),
        "fp16_correct": not hf_res.error and results_match(gold, hf_res, om),
        "gguf_correct": not q_res.error and results_match(gold, q_res, om),
        "fp16_s": round(hf_s, 1),
        "gguf_s": round(q_s, 1),
        "fp16_sql": hf_sql,
        "gguf_sql": q_sql,
    }
    check.append(row)
    print(json.dumps({k: v for k, v in row.items() if not k.endswith("_sql")}, ensure_ascii=False))
    if not row["same_sql"]:
        print("  fp16:", hf_sql.replace("\n", " ")[:300])
        print("  gguf:", q_sql.replace("\n", " ")[:300])

summary = {
    "n": len(check),
    "same_sql": sum(r["same_sql"] for r in check),
    "same_result": sum(r["same_result"] for r in check),
    "fp16_correct": sum(r["fp16_correct"] for r in check),
    "gguf_correct": sum(r["gguf_correct"] for r in check),
    "same_prompt": same_prompt,
}
print(
    f"SQL idêntica: {summary['same_sql']}/{summary['n']} | resultado igual na execução: "
    f"{summary['same_result']}/{summary['n']} | certas (gold): fp16 {summary['fp16_correct']}, "
    f"gguf {summary['gguf_correct']}"
)
del model
gc.collect()

In [ ]:
# 6. export-info.json e model card.
info = {
    "base_model": BASE_MODEL_ID,
    "base_revision": BASE_REVISION,
    "adapter_id": ADAPTER_ID,
    "adapter_revision": ADAPTER_REVISION,
    "repo_ref": REPO_REF,
    "repo_commit": REPO_COMMIT,
    "llama_cpp_ref": LLAMA_CPP_REF,
    "llama_cpp_commit": LLAMA_CPP_COMMIT,
    "llama_cpp_python": llama_cpp.__version__,
    "transformers": transformers.__version__,
    "torch": torch.__version__,
    "device": DEVICE,
    "quantization": QUANT,
    "gguf_file": GGUF_NAME,
    "gguf_sha256": GGUF_SHA256,
    **sizes,
    "timings_s": timings,
    "check": {"dataset": CHECK_DATASET, **summary, "items": check},
}
(Path(OUT_DIR) / "export-info.json").write_text(
    json.dumps(info, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)

CARD = f"""---
license: apache-2.0
language:
  - pt
base_model: {BASE_MODEL_ID}
pipeline_tag: text-generation
library_name: gguf
tags:
  - gguf
  - text-to-sql
  - duckdb
  - llama-cpp
  - qwen2.5
  - anp
datasets:
  - belasoares/tanque-cheio
---

# tanque-cheio · Qwen2.5-Coder-1.5B ajustado, em GGUF {QUANT}

Modelo de text-to-SQL em português sobre os preços de combustíveis da ANP: recebe uma
pergunta e devolve uma consulta SQL para DuckDB. É o adapter v2
[`{ADAPTER_ID}`](https://huggingface.co/{ADAPTER_ID}) (commit `{ADAPTER_REVISION[:7]}`)
fundido no [`{BASE_MODEL_ID}`](https://huggingface.co/{BASE_MODEL_ID}) (commit
`{BASE_REVISION[:7]}`), salvo em fp16 e convertido para GGUF {QUANT} com o llama.cpp
`{LLAMA_CPP_REF}`. Roda em CPU com o `llama-cpp-python` e é o modelo da demo
<https://huggingface.co/spaces/belasoares/tanque-cheio>.

| Arquivo | Tamanho | SHA-256 |
|---|---|---|
| `{GGUF_NAME}` | {sizes["gguf_q_gib"]} GiB | `{GGUF_SHA256}` |

## Resultados

No teste do projeto (150 perguntas), o modelo fundido em fp16 acertou **80,7%** pela
execução (EX), empatado com o Qwen2.5-Coder-32B-Instruct sem ajuste. Esta versão em
{QUANT} não foi avaliada no teste; numa conferência com {summary["n"]} perguntas do dev2, a
SQL foi idêntica à do fp16 em {summary["same_sql"]}/{summary["n"]} e o resultado bateu na
execução em {summary["same_result"]}/{summary["n"]}.

## Uso

O modelo espera o prompt único do projeto (`tanque.prompting.build_prompt`), com o
esquema do banco e as convenções das respostas:

```python
from llama_cpp import Llama
from tanque.prompting import build_prompt, extract_sql

llm = Llama(model_path="{GGUF_NAME}", n_ctx=2048, verbose=False)
out = llm.create_chat_completion(
    messages=build_prompt("O etanol compensou em São Carlos (SP) no último mês?"),
    temperature=0.0,
    max_tokens=384,
)
print(extract_sql(out["choices"][0]["message"]["content"]))
```

Código, dados e avaliação: <https://github.com/belasoaress/tanque-cheio>.

## Licença e limites

Pesos sob Apache-2.0, como o modelo base. Os dados são da ANP (2024-10 a 2026-09), com
atribuição à ANP. A SQL pode estar errada: confira a consulta antes de usar o número.
"""
(Path(OUT_DIR) / "README.md").write_text(CARD, encoding="utf-8")
print(sorted(p.name for p in Path(OUT_DIR).iterdir()))

In [ ]:
# 7. Publica no HF só com PUSH_TO_HUB = True e o secret HF_TOKEN (Add-ons → Secrets).
if PUSH_TO_HUB:
    from huggingface_hub import HfApi
    from kaggle_secrets import UserSecretsClient

    api = HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))
    api.create_repo(HF_REPO, repo_type="model", exist_ok=True)
    commit = api.upload_folder(
        repo_id=HF_REPO,
        repo_type="model",
        folder_path=OUT_DIR,
        allow_patterns=[GGUF_NAME, "README.md", "export-info.json"],
        commit_message=f"GGUF {QUANT} do ft2 ({ADAPTER_REVISION[:7]}), llama.cpp {LLAMA_CPP_REF}",
    )
    print("publicado:", commit.commit_url)
    print("GGUF_REVISION para o app:", commit.oid)
else:
    print("PUSH_TO_HUB desligado: nada foi publicado (docs/demo.md, passo a).")